# Scenario 2: A cross-functional team with one data scientist working on an ML model

MLflow setup:
- tracking server: yes, local server
- backend store: sqlite database
- artifacts store: local filesystem

The experiments can be explored locally by accessing the local tracking server.

To run this example you need to launch the mlflow server locally by running the following command in your terminal:

```bash
uvx mlflow==3.10.1 server --backend-store-uri "sqlite:///$(pwd)/scenario-2-mlflow.db"
```

In [1]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [2]:
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

Tracking URI: http://127.0.0.1:5000


In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1776015499155, experiment_id='0', last_update_time=1776015499155, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment('my-experiment-2')

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {'C': 0.1, 'random_state': 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric('acurracy', accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, name='models')
    print(f'Artifacts URI: {mlflow.get_artifact_uri()}')

2026/04/12 14:38:35 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-2' does not exist. Creating a new experiment.
2026/04/12 14:38:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Artifacts URI: mlflow-artifacts:/1/36402a67ea3c464ea42cd184c14fc7a5/artifacts
🏃 View run able-bird-742 at: http://127.0.0.1:5000/#/experiments/1/runs/36402a67ea3c464ea42cd184c14fc7a5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1776015515153, experiment_id='1', last_update_time=1776015515153, lifecycle_stage='active', name='my-experiment-2', tags={}, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1776015499155, experiment_id='0', last_update_time=1776015499155, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

## Interacting with the model registry

In [6]:
from mlflow.tracking import MlflowClient

client = MlflowClient('http://127.0.0.1:5000')

In [7]:
client.search_registered_models()

[]

In [8]:
client.search_runs(experiment_ids=['1'])[0]

<Run: data=<RunData: metrics={'acurracy': 0.96}, params={'C': '0.1', 'random_state': '42'}, tags={'mlflow.runName': 'able-bird-742',
 'mlflow.source.name': 'scenario-2.ipynb',
 'mlflow.source.type': 'NOTEBOOK',
 'mlflow.user': 'danielwohlgemuth'}>, info=<RunInfo: artifact_uri='mlflow-artifacts:/1/36402a67ea3c464ea42cd184c14fc7a5/artifacts', end_time=1776015518716, experiment_id='1', lifecycle_stage='active', run_id='36402a67ea3c464ea42cd184c14fc7a5', run_name='able-bird-742', start_time=1776015515253, status='FINISHED', user_id='danielwohlgemuth'>, inputs=<RunInputs: dataset_inputs=[], model_inputs=[]>, outputs=<RunOutputs: model_outputs=[<LoggedModelOutput: model_id='m-6960924e354b4e6c8806a0825c7a23af', step=0>]>>

In [9]:
client.search_logged_models(experiment_ids=['1'])[0]

LoggedModel(artifact_location='mlflow-artifacts:/1/models/m-6960924e354b4e6c8806a0825c7a23af/artifacts', creation_timestamp=1776015515339, experiment_id='1', last_updated_timestamp=1776015518705, model_id='m-6960924e354b4e6c8806a0825c7a23af', model_type='', model_uri='models:/m-6960924e354b4e6c8806a0825c7a23af', name='models', source_run_id='36402a67ea3c464ea42cd184c14fc7a5', status=<LoggedModelStatus.READY: 'READY'>, status_message='')

In [10]:
run = client.search_runs(experiment_ids=['1'])[0]
artifact_uri = run.info.artifact_uri
mlflow.register_model(model_uri=artifact_uri, name='iris-classifier')

Successfully registered model 'iris-classifier'.
2026/04/12 14:38:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1776015530973, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1776015530973, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='', run_link='', source='mlflow-artifacts:/1/36402a67ea3c464ea42cd184c14fc7a5/artifacts', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>